In [1]:
import numpy as np
import os
import itertools
import logging
import matplotlib.pyplot as plt
import MDAnalysis as MDA
import torch
import torch.optim as optim
import torch.nn.functional as F
from argparse import ArgumentParser
from torch.distributions import MultivariateNormal
import sys
sys.path.append("../")
import pymbar
from config import get_cfg_defaults
from nf.flows import *
from nf.models import NormalizingFlowModel
from nf.base import EinsteinCrystal
import nf.utils as util
import lammps
%load_ext autoreload

/home/sherryli/xsli/softwares/anaconda3/envs/sherry/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def read_input(dir):
    cfg = get_cfg_defaults()
    cfg.merge_from_file(dir)
    cfg.freeze()
    print(cfg)
    return cfg

In [3]:
def setup_model(cfg):
    if cfg.dataset.rho is not None:
        B=(cfg.dataset.nparticles/(8*cfg.dataset.rho))**(1/3)
    else:
        B=cfg.dataset.ncellx*cfg.dataset.cell_len/2
    boxlength=2*B
    print("B:",B)
    N=cfg.dataset.nparticles*3
    logging.basicConfig(level=logging.DEBUG)
    logger = logging.getLogger(__name__)  
    flows = [eval(cfg.flow.type)(dim=N, K=cfg.flow.nsplines, B=B,hidden_dim=cfg.flow.hidden_dim,device=cfg.device) for _ in range(cfg.flow.nlayers)]
    if cfg.prior.type=="lattice":
        prior = EinsteinCrystal(cfg.prior.lattice_dir, alpha=cfg.prior.alpha,device=cfg.device)
    elif cfg.prior.type=="normal":
        prior = MultivariateNormal(torch.zeros(N).to(cfg.device), torch.eye(N).to(cfg.device))
    model = NormalizingFlowModel(prior, flows).to(cfg.device)
    optimizer = optim.Adam(model.parameters(), lr=cfg.train_parameters.learning_rate)
    #scheduler = torch. optim.lr_scheduler.ExponentialLR(optimizer, cfg.train_parameters.lr_scheduler_gamma)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, cfg.train_parameters.max_epochs)
    training_data = util.load_position(cfg.dataset.input_dir).to(cfg.device)
    with open(cfg.output.pos_dir, 'w'):
        pass
    if not(os.path.exists(cfg.output.model_dir)):
        os.mkdir(cfg.output.model_dir)
    return model,optimizer,scheduler,training_data,logger,boxlength

In [4]:
def U(positions, boxlength):
    return(torch.mean(torch.Tensor([util.LJ_potential(pos.reshape((-1,3)), boxlength, cutoff=2.7) for pos in positions])))


In [5]:
def reverseKL(cfg,model,nsamples,boxlength):
    z = model.prior.sample((nsamples,))
    x, log_det = model.inverse(z)
    log_prob = model.prior.log_prob(z)-log_det
    Ulj=U(x,boxlength)
    return Ulj/cfg.dataset.kT+torch.mean(log_prob)

In [6]:
def train(cfg,model,optimizer,scheduler,training_data,logger,boxlength):
    with open("base.xyz", 'w'):
            pass
    losses=[]
    max_logprob=140
    lamb=0.5
    for i in range(cfg.train_parameters.max_epochs):
        optimizer.zero_grad()
        x = util.subsample(training_data,nsamples=cfg.train_parameters.batch_size,device=cfg.device)
        z, prior_logprob, log_det = model(x)
        util.write_coord("base.xyz",z.detach(),cfg.dataset.nparticles,boxlength)
        #x1, log_det1 =model.inverse(z)
        #print("check the map is invertible", torch.linalg.norm(x1-x), torch.linalg.norm(log_det1+log_det))
        logprob = prior_logprob + log_det
        forward_loss=-torch.mean(logprob)
        #if i>4000:
            #loss = forward_loss*lamb + (1-lamb)*reverseKL(cfg,model, cfg.train_parameters.batch_size,boxlength)
        #else:
        loss = forward_loss#-U(x,boxlength)/cfg.dataset.kT
        losses.append(loss.mean().data)
        loss.backward()
        optimizer.step()
        scheduler.step()
        if i % cfg.train_parameters.output_freq == 0:
            logger.info(f"Iter: {i}\t" +
                        f"Loss: {loss.mean().data:.2f}\t" +
                        f"Logprob: {logprob.mean().data:.2f}\t" +
                        f"Prior: {prior_logprob.mean().data:.2f}\t" +
                        f"LogDet: {log_det.mean().data:.2f}")
            samples,_,z = model.sample(1)
            util.write_coord(cfg.output.pos_dir,samples,cfg.dataset.nparticles,boxlength)
            if (i>1200) and (-forward_loss>max_logprob):
                max_logprob=-forward_loss
                torch.save({"model":model.state_dict(),"optim": optimizer.state_dict(),
                            "loss":losses},cfg.output.model_dir+cfg.dataset.name+'%d.pth'% (i//cfg.train_parameters.output_freq))

In [7]:
cfg=read_input("input/fe.yaml")
model,optimizer,scheduler,training_data,logger,boxlength = setup_model(cfg)

dataset:
  cell_len: 2.9115
  input_dir: md_data/fe_400K.xyz
  kT: 0.137877332192
  name: Fe
  ncellx: 3
  ncelly: 3
  ncellz: 3
  nparticles: 54
  rho: None
device: cuda:0
flow:
  hidden_dim: 324
  nlayers: 3
  nsplines: 32
  type: NSF_AR
output:
  model_dir: saved_models/
  pos_dir: ./fe_positions_during_training.xyz
prior:
  alpha: 300
  lattice_dir: md_data/fe_ref.xyz
  type: lattice
train_parameters:
  batch_size: 100
  learning_rate: 0.0001
  lr_scheduler_gamma: 0.999
  max_epochs: 4000
  output_freq: 100
B: 4.36725


In [ ]:
train(cfg,model,optimizer,scheduler,training_data,logger,boxlength)

INFO:__main__:Iter: 0	Loss: 90.88	Logprob: -82.47	Prior: -79.54	LogDet: -2.93
INFO:__main__:Iter: 100	Loss: -175.03	Logprob: 183.44	Prior: 228.41	LogDet: -44.96
INFO:__main__:Iter: 200	Loss: -176.34	Logprob: 184.75	Prior: 227.79	LogDet: -43.04
INFO:__main__:Iter: 300	Loss: -180.30	Logprob: 188.71	Prior: 229.91	LogDet: -41.19
INFO:__main__:Iter: 400	Loss: -184.77	Logprob: 193.18	Prior: 231.49	LogDet: -38.32
INFO:__main__:Iter: 500	Loss: -185.53	Logprob: 193.95	Prior: 227.50	LogDet: -33.55
INFO:__main__:Iter: 600	Loss: -190.49	Logprob: 198.90	Prior: 229.08	LogDet: -30.17
INFO:__main__:Iter: 700	Loss: -192.12	Logprob: 200.54	Prior: 229.67	LogDet: -29.13
INFO:__main__:Iter: 800	Loss: -193.63	Logprob: 202.04	Prior: 229.84	LogDet: -27.79
INFO:__main__:Iter: 900	Loss: -194.47	Logprob: 202.88	Prior: 228.70	LogDet: -25.82
INFO:__main__:Iter: 1000	Loss: -194.62	Logprob: 203.03	Prior: 226.81	LogDet: -23.78
INFO:__main__:Iter: 1100	Loss: -197.78	Logprob: 206.19	Prior: 230.63	LogDet: -24.44
INFO:__

In [14]:
def generate_from_nf(model, prior, nsamples=50):
    #x, log_det ,z = model.sample(nsamples)
    z=prior.sample((nsamples,))
    x, log_det = model.inverse(z)
    log_px=prior.log_prob(z)-log_det
    return x.data, log_px.data

def load_md_data(cfg,dir,model,prior,boxlength):
    traj = MDA.coordinates.XYZ.XYZReader(dir)
    pos = torch.from_numpy(
        np.array([np.array(traj[i]) for i in range(len(traj))])).to(cfg.device)
    pe=torch.Tensor([util.LJ_potential(pos[i], boxlength,cutoff=2.7) for i in range(len(pos))])/cfg.dataset.kT
    z,_,log_det=model.forward(pos.reshape(len(traj),-1))
    q_nf=prior.log_prob(z)-log_det
    return pos,q_nf.data,-pe.data

In [ ]:
file_dir="structures/lj_test.xyz"
output_dir="generated_configs.xyz"
with open(output_dir,"w"):
    pass
nf = torch.load("saved_models/LJ35.pth")
np.savetxt("losses.dat",torch.Tensor(nf["loss"]).cpu().numpy())
model.load_state_dict(nf["model"])
samples,_,z = model.sample(100)

sample_prior = EinsteinCrystal(cfg.prior.lattice_dir, alpha=1000)
z = sample_prior.sample((100,))
samples, log_det = model.inverse(z)

util.write_coord(output_dir,samples.data,cfg.dataset.nparticles,boxlength)

traj0,q00=generate_from_nf(model,sample_prior, nsamples=8000)
q00=q00.cpu().numpy()
traj0=traj0.cpu().reshape(-1,cfg.dataset.nparticles,3)
q01=np.array([util.LJ_potential(traj0[i], boxlength,cutoff=2.7) for i in range(len(traj0))])
q01=-q01/cfg.dataset.kT
Q=[]
Q.append(np.transpose(np.vstack((q00,q01))))
traj1,q10,q11=load_md_data(cfg,file_dir,model,sample_prior,boxlength)
q10=q10.cpu().numpy()
q11=q11.cpu().numpy()
Q.append(np.transpose(np.vstack((q10,q11))))
with open("Q0.dat","w"):
    pass
with open("Q0.dat", "ab") as f:
    np.savetxt(f, Q[0])
np.savetxt("Q1.dat",Q[1])

In [20]:
Nk=np.array([len(Q[0]),len(Q[1])])
u=np.vstack((-Q[0],-Q[1])).transpose()
print(u.shape)
mbar=pymbar.mbar.MBAR(u,Nk)
normconst=mbar.getFreeEnergyDifferences(return_dict=True)
print(normconst)
u_sq=np.vstack((Q[0]**2,Q[1]**2)).transpose()
heat_cap=mbar.computeExpectations(u_sq, return_dict=True)
print(heat_cap)

(2, 17999)
{'Delta_f': array([[   0.        ,  192.92417723],
       [-192.92417723,    0.        ]]), 'dDelta_f': array([[0.        , 0.04225576],
       [0.04225576, 0.        ]])}
{'mu': array([ 1235.20692407, 22483.88222075]), 'sigma': array([  5.82680805, 108.34601803])}


In [12]:
def metropolize(cfg,x,burnin=20):
    nsamples=x.size(dim=0)
    index=[False for i in range(nsamples)]
    frame=x[0].reshape(cfg.dataset.nparticles,3)
    energy=util.LJ_potential(frame, boxlength,cutoff=2.7)
    for i in range(nsamples):
        new_frame=x[i].reshape(cfg.dataset.nparticles,3)
        new_energy=util.LJ_potential(new_frame, boxlength,cutoff=2.7)
        acc_prob=torch.exp(energy-new_energy)
        if torch.rand(1)<acc_prob:
            frame=new_frame
            energy=new_energy
            if i>burnin:
                index[i]=True
    return index

In [22]:
nbatches=1
batchsize=4000
pos=[]
logp=[]
pot=[]
for i in range(nbatches):
    z = sample_prior.sample((batchsize,))
    x, log_det = model.inverse(z)
    index =metropolize(cfg,x.data)
    pos.append(x.data[index])
    logp.append(sample_prior.log_prob(z[index])-log_det[index])
    pot.append(torch.Tensor([util.LJ_potential(x[i].reshape(-1,3), boxlength,cutoff=2.7) for i in np.arange(batchsize)[index]])/cfg.dataset.kT)


In [ ]:
lmp=lammps.lammps()
lmp.command("read_dump md_data/fe_400K.xyz")